# 🤖 Multi-Exchange Scalping Bot — BTC & SOL
**Strategy:** Smart Money Concepts (MSB + Order Blocks + FVG) + EMA + RSI + ATR stops

---
## Architecture

```
┌─────────────────────────────┐      ┌──────────────────────────────┐
│   DATA SOURCE (Cell 2)      │      │   EXECUTION EXCHANGE (Cell 2) │
│                             │      │                              │
│  • Binance  (deepest OHLCV) │  →   │  • Kraken   (spot trading)   │
│  • Kraken   (native)        │      │  • Binance  (spot/futures)   │
│  • Forex    (OANDA/yfinance)│      │  • Paper    (no exchange)    │
└─────────────────────────────┘      └──────────────────────────────┘
```

## Notebook phases

| Phase | Cells | Description |
|-------|-------|-------------|
| **1 — Setup** | 1–4 | Install, configure data source + execution exchange |
| **2 — Connect** | 5 | Connect to both exchanges and validate pairs |
| **3 — Indicators & signals** | 6–7 | Indicator functions + strategy engine |
| **4 — Backtest** | 8–9 | Fetch history from data source, run backtest |
| **5 — Live/Paper loop** | 10–12 | Execute on chosen exchange |

> ⚠️ **Live trading is OFF by default.** API keys are only needed for the execution exchange.  
> Data sources (Binance public, Forex) require no credentials.

---
## Phase 1 — Setup

In [ ]:
# ── CELL 1 — Install dependencies ───────────────────────────────────────────
import sys
!{sys.executable} -m pip install ccxt pandas numpy matplotlib yfinance python-dotenv --quiet
print('✅ Dependencies installed')

In [ ]:
# ── CELL 2 — Configuration ───────────────────────────────────────────────────
#
# DATA SOURCE  → where to pull OHLCV history and live candles from
# EXEC EXCHANGE → where to place actual orders
#
# These are INDEPENDENT. Common setups:
#   DATA_SOURCE='binance'  + EXEC_EXCHANGE='kraken'   ← recommended
#   DATA_SOURCE='binance'  + EXEC_EXCHANGE='binance'  ← all-Binance
#   DATA_SOURCE='kraken'   + EXEC_EXCHANGE='kraken'   ← all-Kraken
#   DATA_SOURCE='forex'    + EXEC_EXCHANGE='paper'    ← forex paper mode

# ── Data source ───────────────────────────────────────────────────────────────
# 'binance' → public API, no key needed, deep history, USDT pairs only
# 'kraken'  → public API, no key needed for data, USD + USDT pairs
# 'forex'   → yfinance (Yahoo Finance), no key needed, FX + crypto pairs
DATA_SOURCE = 'binance'

# Pairs to fetch data for (must match the format of DATA_SOURCE)
# Binance : 'BTC/USDT', 'SOL/USDT', 'ETH/USDT'
# Kraken  : 'BTC/USD',  'SOL/USD',  'BTC/USDT', 'SOL/USDT'
# Forex   : 'EUR/USD',  'GBP/USD',  'BTC-USD'  (yfinance ticker format)
DATA_PAIRS = ['BTC/USDT', 'SOL/USDT']

# ── Execution exchange ────────────────────────────────────────────────────────
# 'kraken'  → execute on Kraken (requires API key)
# 'binance' → execute on Binance (requires API key)
# 'paper'   → no exchange needed, simulate locally
EXEC_EXCHANGE = 'paper'    # <── change to 'kraken' or 'binance' when ready

# Pairs to trade on the execution exchange
# Must be valid symbols on that exchange (may differ from DATA_PAIRS format)
# Kraken  : 'BTC/USD', 'SOL/USD'
# Binance : 'BTC/USDT', 'SOL/USDT'
# Paper   : any label you want (no real exchange)
EXEC_PAIRS = ['BTC/USDT', 'SOL/USDT']

# ── Pair mapping: data source → execution exchange ────────────────────────────
# Maps each DATA_PAIRS symbol to the corresponding EXEC_PAIRS symbol.
# Needed because Binance uses BTC/USDT while Kraken uses BTC/USD.
# If your data and exec pairs are identical, just make them the same.
PAIR_MAP = {
    'BTC/USDT': 'BTC/USDT',   # data symbol → exec symbol
    'SOL/USDT': 'SOL/USDT',
    'BTC/USD':  'BTC/USD',
    'SOL/USD':  'SOL/USD',
    # Binance data → Kraken execution example:
    # 'BTC/USDT': 'BTC/USD',
    # 'SOL/USDT': 'SOL/USD',
}

# ── Trading mode ──────────────────────────────────────────────────────────────
# 'paper' → simulate trades, zero risk (DEFAULT)
# 'live'  → real orders on EXEC_EXCHANGE (use with caution)
TRADING_MODE = 'paper'

# ── Execution exchange credentials ───────────────────────────────────────────
# Only needed when EXEC_EXCHANGE = 'kraken' or 'binance' AND TRADING_MODE = 'live'
# Public data endpoints (history, live prices) never need credentials.
EXEC_API_KEY    = 'your_api_key_here'
EXEC_API_SECRET = 'your_api_secret_here'

# ── Strategy parameters ───────────────────────────────────────────────────────
STRATEGY_CONFIG = {
    'risk_percent':  1.0,    # % of equity risked per trade
    'rr':            2.0,    # risk-to-reward ratio
    'use_trailing':  False,  # trailing stop (enable after validating R:R)
    'trail_percent': 0.6,    # trailing stop distance as % of price
    'commission':    0.04,   # Binance futures / Kraken — adjust to your exchange
    'atr_multiplier': 1.2,   # stop_dist = ATR * this value
    'vol_multiplier': 1.5,   # volume must be this × vol_ma to confirm break
    'rsi_long_min':  55,     # RSI threshold for long entries
    'rsi_short_max': 45,     # RSI threshold for short entries
}

# ── Bot / backtest settings ───────────────────────────────────────────────────
LOOP_INTERVAL    = 60      # seconds between live scan cycles
PAPER_EQUITY     = 10000   # starting virtual USD
BACKTEST_CANDLES = 1000    # candles to fetch for backtest (Binance max per request: 1000)
BACKTEST_DAYS    = 7       # if > 0, fetches paginated history for this many days
CANDLE_LIMIT     = 200     # candles fetched per live loop cycle

# ── Banner ────────────────────────────────────────────────────────────────────
print(f'Data source      : {DATA_SOURCE.upper()}')
print(f'Data pairs       : {DATA_PAIRS}')
print(f'Exec exchange    : {EXEC_EXCHANGE.upper()}')
print(f'Exec pairs       : {EXEC_PAIRS}')
print(f'Trading mode     : {"📄 PAPER" if TRADING_MODE == "paper" else "⚡ LIVE"}')
print()
print('Pair mapping:')
for d, e in PAIR_MAP.items():
    if d in DATA_PAIRS:
        print(f'  {d:12s} (data) → {e:12s} (exec)')

In [ ]:
# ── CELL 3 — Indicator functions ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta
import time, warnings
warnings.filterwarnings('ignore')

def ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=period - 1, adjust=False).mean()
    avg_loss = loss.ewm(com=period - 1, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def atr(high, low, close, period=10):
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(com=period - 1, adjust=False).mean()

def sma(series, period):
    return series.rolling(window=period).mean()

def highest(series, period):
    return series.rolling(window=period).max()

def lowest(series, period):
    return series.rolling(window=period).min()

print('✅ Indicator functions ready')

In [ ]:
# ── CELL 4 — Strategy signal engine ──────────────────────────────────────────
# All configurable thresholds now read from STRATEGY_CONFIG so you can
# tune them in Cell 2 without touching this cell.

def compute_signals(df, config):
    df = df.copy()
    atr_mult  = config.get('atr_multiplier', 1.2)
    vol_mult  = config.get('vol_multiplier', 1.5)
    rsi_long  = config.get('rsi_long_min',  55)
    rsi_short = config.get('rsi_short_max', 45)

    # 1. Indicators
    df['ema9']   = ema(df['close'], 9)
    df['ema21']  = ema(df['close'], 21)
    df['rsi']    = rsi(df['close'], 14)
    df['atr']    = atr(df['high'], df['low'], df['close'], 10)
    df['vol_ma'] = sma(df['volume'], 15)

    # 2. Market Structure Breaks
    rolling_high = highest(df['high'].shift(1), 15).shift(1)
    rolling_low  = lowest(df['low'].shift(1), 15).shift(1)
    df['bull_break'] = (
        (df['high'] > rolling_high) &
        (df['close'] > df['open']) &
        (df['volume'] > df['vol_ma'] * vol_mult)
    )
    df['bear_break'] = (
        (df['low'] < rolling_low) &
        (df['close'] < df['open']) &
        (df['volume'] > df['vol_ma'] * vol_mult)
    )

    # 3. Order Blocks
    bull_chg = df['bull_break'] & ~df['bull_break'].shift(1).fillna(False)
    bear_chg = df['bear_break'] & ~df['bear_break'].shift(1).fillna(False)
    df['bull_ob'] = pd.Series(
        np.where(bull_chg, df['low'].shift(1), np.nan), index=df.index).ffill()
    df['bear_ob'] = pd.Series(
        np.where(bear_chg, df['high'].shift(1), np.nan), index=df.index).ffill()

    # 4. Fair Value Gaps
    df['fvg_bull'] = (df['low'].shift(2)  > df['high']) & (df['close'] > df['open'])
    df['fvg_bear'] = (df['high'].shift(2) < df['low'])  & (df['close'] < df['open'])

    # 5. Entry signals
    df['long_signal'] = (
        (df['bull_break'] | df['fvg_bull']) &
        (df['close'] > df['bull_ob']) &
        (df['close'] > df['ema9']) &
        (df['ema9']  > df['ema21']) &
        (df['rsi']   > rsi_long) &
        (df['rsi']   < 80)
    )
    df['short_signal'] = (
        (df['bear_break'] | df['fvg_bear']) &
        (df['close'] < df['bear_ob']) &
        (df['close'] < df['ema9']) &
        (df['ema9']  < df['ema21']) &
        (df['rsi']   < rsi_short) &
        (df['rsi']   > 20)
    )

    # 6. Exit levels
    df['stop_dist']    = df['atr'] * atr_mult
    df['take_profit']  = df['stop_dist'] * config['rr']
    df['long_sl']      = df['close'] - df['stop_dist']
    df['long_tp']      = df['close'] + df['take_profit']
    df['short_sl']     = df['close'] + df['stop_dist']
    df['short_tp']     = df['close'] - df['take_profit']
    if config['use_trailing']:
        df['trail_points'] = df['close'] * (config['trail_percent'] / 100)
        df['trail_offset'] = df['stop_dist'] * 0.5
    else:
        df['trail_points'] = np.nan
        df['trail_offset'] = np.nan

    last = df.iloc[-1]
    signal = {k: (bool(last[k]) if k.endswith('signal') else float(last[k]))
              for k in ['long_signal','short_signal','close',
                        'long_sl','long_tp','short_sl','short_tp',
                        'trail_points','trail_offset','atr','rsi','ema9','ema21']}
    return df, signal

print('✅ Strategy engine ready')

---
## Phase 2 — Connect to exchanges
*Cell 5 connects to both the data source and execution exchange independently.*

In [ ]:
# ── CELL 5 — Connect to data source + execution exchange ─────────────────────
#
# DATA SOURCE   → public endpoints only, no credentials needed
# EXEC EXCHANGE → credentials only required for live trading
#
# Both connections are independent. If DATA_SOURCE == EXEC_EXCHANGE, we
# reuse the same connection object to avoid duplicate API calls.

import ccxt
import yfinance as yf

# ── Data source connection ────────────────────────────────────────────────────
def connect_data_source(source):
    """
    Returns a connection object for the chosen data source.
    Binance and Kraken use ccxt (no credentials needed for public data).
    Forex uses yfinance — no connection object needed, fetched per-call.
    """
    if source == 'binance':
        conn = ccxt.binance({'enableRateLimit': True})
        conn.load_markets()
        print(f'✅ Data source: Binance ({len(conn.markets)} markets, no API key needed)')
        return conn
    elif source == 'kraken':
        conn = ccxt.kraken({'enableRateLimit': True})
        conn.load_markets()
        print(f'✅ Data source: Kraken ({len(conn.markets)} markets, no API key needed)')
        return conn
    elif source == 'forex':
        print('✅ Data source: Forex / Yahoo Finance (yfinance, no credentials needed)')
        return None   # yfinance is stateless, no connection object
    else:
        raise ValueError(f'Unknown data source: {source}. Choose binance / kraken / forex')


# ── Execution exchange connection ─────────────────────────────────────────────
def connect_exec_exchange(exchange_name, api_key, api_secret):
    """
    Returns an authenticated ccxt connection for order execution.
    In paper mode, returns None — no exchange connection needed.
    """
    if exchange_name == 'paper':
        print('✅ Exec exchange: PAPER (no connection needed)')
        return None
    elif exchange_name == 'kraken':
        conn = ccxt.kraken({
            'apiKey': api_key, 'secret': api_secret, 'enableRateLimit': True})
        conn.load_markets()
        print(f'✅ Exec exchange: Kraken ({len(conn.markets)} markets)')
        return conn
    elif exchange_name == 'binance':
        conn = ccxt.binance({
            'apiKey': api_key, 'secret': api_secret, 'enableRateLimit': True})
        conn.load_markets()
        print(f'✅ Exec exchange: Binance ({len(conn.markets)} markets)')
        return conn
    else:
        raise ValueError(f'Unknown exec exchange: {exchange_name}. Choose kraken / binance / paper')


# ── Instantiate both connections ──────────────────────────────────────────────
data_conn = connect_data_source(DATA_SOURCE)

# Reuse data connection if both point to the same exchange (avoids duplicate loads)
if EXEC_EXCHANGE == DATA_SOURCE and EXEC_EXCHANGE != 'paper':
    exec_conn = data_conn
    print(f'ℹ️  Exec exchange reuses data connection ({EXEC_EXCHANGE})')
else:
    exec_conn = connect_exec_exchange(EXEC_EXCHANGE, EXEC_API_KEY, EXEC_API_SECRET)

# ── Validate exec pairs on execution exchange ─────────────────────────────────
valid_exec_pairs = []
if exec_conn is not None:
    print()
    for pair in EXEC_PAIRS:
        if pair in exec_conn.markets:
            valid_exec_pairs.append(pair)
            print(f'  ✅ {pair} available on {EXEC_EXCHANGE}')
        else:
            print(f'  ⚠️  {pair} NOT found on {EXEC_EXCHANGE} — will be skipped')
    EXEC_PAIRS = valid_exec_pairs
else:
    valid_exec_pairs = EXEC_PAIRS   # paper mode — accept all

# ── Show live balance if using real credentials ───────────────────────────────
if EXEC_EXCHANGE != 'paper' and EXEC_API_KEY != 'your_api_key_here':
    try:
        bal = exec_conn.fetch_balance()
        print()
        print(f'💰 {EXEC_EXCHANGE.capitalize()} account balance:')
        for currency in ['USD', 'USDT', 'XBT', 'BTC', 'SOL', 'ETH']:
            free = bal.get(currency, {}).get('free', 0)
            if free and float(free) > 0:
                print(f'   {currency}: {free}')
    except Exception as e:
        print(f'⚠️  Could not fetch balance: {e}')

---
## Phase 3 — Indicators & signals
*Cells 3–4 above must already be run. This phase has no additional cells — indicators and strategy are already loaded.*

---
## Phase 4 — Backtest
*Cell 6 fetches historical data from your chosen DATA_SOURCE.  
Cell 7 runs the backtest. Both cells finish and return — no loop.*

In [ ]:
# ── CELL 6 — Multi-source historical data fetcher ────────────────────────────
#
# Fetches OHLCV data from whichever DATA_SOURCE is configured in Cell 2.
# Binance and Kraken: uses ccxt with optional pagination for longer histories.
# Forex: uses yfinance (Yahoo Finance) — supports FX pairs, crypto, indices.
#
# All sources return the same DataFrame format:
#   columns : open, high, low, close, volume
#   index   : DatetimeIndex (UTC)

def fetch_ccxt_paginated(conn, symbol, timeframe='1m', days=7, limit_per_req=1000):
    """
    Fetch multiple pages of OHLCV from any ccxt exchange.
    Walks backwards in time to collect `days` worth of 1-min bars.
    Binance: up to 1000 bars/request.  Kraken: up to 720 bars/request.
    """
    all_rows = []
    since_ms  = int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp() * 1000)
    now_ms    = int(datetime.now(timezone.utc).timestamp() * 1000)
    cursor    = since_ms
    pages     = 0

    while cursor < now_ms:
        try:
            raw = conn.fetch_ohlcv(symbol, timeframe=timeframe,
                                   since=cursor, limit=limit_per_req)
        except Exception as e:
            print(f'    Fetch error: {e}')
            break
        if not raw:
            break
        all_rows.extend(raw)
        cursor = raw[-1][0] + 1   # advance cursor past last bar
        pages += 1
        time.sleep(conn.rateLimit / 1000)   # respect rate limit

    print(f'    {pages} pages fetched → {len(all_rows)} raw bars')
    return all_rows


def fetch_from_binance(symbol, days=0, limit=1000):
    """
    Fetch from Binance public API (no key required).
    days > 0 → paginated multi-day fetch
    days = 0 → single request of `limit` bars
    """
    if days > 0:
        raw = fetch_ccxt_paginated(data_conn, symbol, days=days, limit_per_req=1000)
    else:
        raw = data_conn.fetch_ohlcv(symbol, timeframe='1m', limit=limit)
    if not raw:
        return pd.DataFrame()
    df = pd.DataFrame(raw, columns=['timestamp','open','high','low','close','volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df.set_index('timestamp', inplace=True)
    df = df[~df.index.duplicated(keep='last')]   # remove any duplicate timestamps
    return df.iloc[:-1]   # drop still-forming last candle


def fetch_from_kraken(symbol, days=0, limit=720):
    """
    Fetch from Kraken public API (no key required).
    Kraken free tier: ~720 bars per 1m request.
    """
    if days > 0:
        raw = fetch_ccxt_paginated(data_conn, symbol, days=days, limit_per_req=720)
    else:
        raw = data_conn.fetch_ohlcv(symbol, timeframe='1m', limit=limit)
    if not raw:
        return pd.DataFrame()
    df = pd.DataFrame(raw, columns=['timestamp','open','high','low','close','volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df.set_index('timestamp', inplace=True)
    df = df[~df.index.duplicated(keep='last')]
    return df.iloc[:-1]


def fetch_from_forex(symbol, days=7, interval='1m'):
    """
    Fetch from Yahoo Finance via yfinance.
    Supports FX pairs (EURUSD=X), crypto (BTC-USD), indices (^GSPC).
    Symbol format: 'EUR/USD' → 'EURUSD=X', 'BTC/USDT' → 'BTC-USD'
    For 1m data, Yahoo Finance only provides the last 7 days max.

    Interval options: '1m', '2m', '5m', '15m', '30m', '60m', '1d'
    """
    # Auto-convert common symbol formats to yfinance tickers
    ticker = symbol
    if '/' in symbol:
        base, quote = symbol.split('/')
        if quote in ('USD', 'USDT'):
            # Crypto: BTC/USD → BTC-USD
            ticker = f'{base}-USD'
        else:
            # FX: EUR/USD → EURUSD=X
            ticker = f'{base}{quote}=X'

    period_map = {1: '1d', 2: '2d', 3: '3d', 5: '5d', 7: '7d'}
    period = period_map.get(days, '7d')

    print(f'    yfinance ticker: {ticker}  period={period}  interval={interval}')
    raw = yf.download(ticker, period=period, interval=interval,
                      auto_adjust=True, progress=False)
    if raw.empty:
        return pd.DataFrame()

    # Normalise column names (yfinance returns multi-level columns sometimes)
    raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                   for c in raw.columns]
    raw.index = pd.to_datetime(raw.index, utc=True)
    raw.index.name = 'timestamp'

    # yfinance may not have 'volume' for FX — fill with zeros so strategy runs
    if 'volume' not in raw.columns:
        raw['volume'] = 0.0

    return raw[['open','high','low','close','volume']].iloc[:-1]


def fetch_history(symbol, source=None, days=None, limit=None):
    """
    Unified history fetcher — routes to the correct provider based on DATA_SOURCE.
    Falls back to Cell 2 defaults if days/limit not specified.
    """
    source = source or DATA_SOURCE
    days   = days   if days   is not None else BACKTEST_DAYS
    limit  = limit  if limit  is not None else BACKTEST_CANDLES

    print(f'  [{source.upper()}] Fetching {symbol}...', end=' ')
    if source == 'binance':
        df = fetch_from_binance(symbol, days=days, limit=limit)
    elif source == 'kraken':
        df = fetch_from_kraken(symbol, days=days, limit=limit)
    elif source == 'forex':
        df = fetch_from_forex(symbol, days=max(days, 1))
    else:
        raise ValueError(f'Unknown source: {source}')

    if not df.empty:
        span = (f"{df.index[0].strftime('%Y-%m-%d %H:%M')} → "
                f"{df.index[-1].strftime('%Y-%m-%d %H:%M')} UTC")
        print(f'{len(df)} bars  ({span})')
    else:
        print('NO DATA')
    return df


def fetch_live_candles(symbol, source=None, limit=None):
    """
    Fetch the latest candles for the live trading loop.
    Same routing logic as fetch_history but uses CANDLE_LIMIT.
    Always fetches the most recent bars (no `since` pagination).
    """
    source = source or DATA_SOURCE
    limit  = limit  or CANDLE_LIMIT
    try:
        if source == 'binance':
            return fetch_from_binance(symbol, days=0, limit=limit)
        elif source == 'kraken':
            return fetch_from_kraken(symbol, days=0, limit=limit)
        elif source == 'forex':
            return fetch_from_forex(symbol, days=1)
    except Exception as e:
        print(f'  ⚠️  Live fetch error {symbol}: {e}')
        return pd.DataFrame()


# ── Run historical fetch for all data pairs ───────────────────────────────────
print(f'📥 Fetching historical data from {DATA_SOURCE.upper()}...')
print(f'   Pairs: {DATA_PAIRS}')
print(f'   Window: {BACKTEST_DAYS} days  ({BACKTEST_CANDLES} bars max per request)\n')

historical_data = {}   # { data_symbol: DataFrame }
for pair in DATA_PAIRS:
    df = fetch_history(pair)
    if not df.empty:
        historical_data[pair] = df

print(f'\n✅ {len(historical_data)}/{len(DATA_PAIRS)} pairs fetched successfully')

In [ ]:
# ── CELL 7 — Backtest engine + performance report ────────────────────────────
# Bar-by-bar simulation on the data fetched in Cell 6.
# No loop — runs once and returns.

def run_backtest(df_raw, config, symbol=''):
    if df_raw.empty or len(df_raw) < 50:
        print(f'  ⚠️  Not enough data for {symbol} ({len(df_raw)} bars)')
        return None
    df, _ = compute_signals(df_raw, config)
    commission = config['commission'] / 100
    trades, equity, position = [], [100.0], None

    for i in range(len(df)):
        row = df.iloc[i]
        eq  = equity[-1]

        if position is not None:
            sl, tp, d = position['sl'], position['tp'], position['direction']
            # Update trailing stop
            if config['use_trailing']:
                t, o = position['trail_points'], position['trail_offset']
                if d == 'long'  and (row['high'] - position['entry']) >= t:
                    sl = max(sl, row['high'] - o)
                elif d == 'short' and (position['entry'] - row['low']) >= t:
                    sl = min(sl, row['low'] + o)
                position['sl'] = sl
            # Check exits (TP priority)
            tp_hit   = (d == 'long'  and row['high'] >= tp) or (d == 'short' and row['low']  <= tp)
            stop_hit = (d == 'long'  and row['low']  <= sl) or (d == 'short' and row['high'] >= sl)
            if tp_hit or stop_hit:
                exit_price = tp if tp_hit else sl
                gross = (exit_price - position['entry']) / position['entry'] if d == 'long' \
                        else (position['entry'] - exit_price) / position['entry']
                net_pct = gross - commission
                equity.append(eq * (1 + net_pct))
                trades.append({
                    'symbol': symbol, 'direction': d,
                    'entry_time': position['entry_time'], 'exit_time': df.index[i],
                    'entry_price': position['entry'], 'exit_price': exit_price,
                    'result': 'TP' if tp_hit else 'SL',
                    'pnl_pct': round((net_pct) * 100, 4),
                    'equity': round(equity[-1], 4),
                })
                position = None
            else:
                equity.append(eq)
        else:
            equity.append(eq)

        if position is None and (row['long_signal'] or row['short_signal']):
            d  = 'long' if row['long_signal'] else 'short'
            sl = float(row['long_sl']  if d == 'long' else row['short_sl'])
            tp = float(row['long_tp']  if d == 'long' else row['short_tp'])
            entry = float(row['close']) * (1 + commission if d == 'long' else 1 - commission)
            position = {'direction': d, 'entry': entry, 'entry_time': df.index[i],
                        'sl': sl, 'tp': tp,
                        'trail_points': float(row['trail_points']),
                        'trail_offset': float(row['trail_offset'])}

    return {'trades': trades, 'equity': equity[:len(df)], 'df': df}


def print_report(trades, symbol):
    if not trades:
        print(f'  {symbol}: no trades generated')
        return
    t    = pd.DataFrame(trades)
    wins = t[t['pnl_pct'] > 0]
    loss = t[t['pnl_pct'] <= 0]
    tp_c = (t['result'] == 'TP').sum()
    sl_c = (t['result'] == 'SL').sum()
    wr   = len(wins) / len(t) * 100
    pf   = abs(wins['pnl_pct'].sum() / loss['pnl_pct'].sum()) if len(loss) > 0 else float('inf')
    peak = t['equity'].iloc[0]
    maxdd = max(((peak := max(peak, eq)) - eq) / peak * 100 for eq in t['equity'])
    print(f'  ┌─ {symbol} (data: {DATA_SOURCE.upper()} → exec: {EXEC_EXCHANGE.upper()}) ' + '─'*10)
    print(f'  │  Trades       : {len(t):>5}   (TP: {tp_c} / SL: {sl_c})')
    print(f'  │  Win rate     : {wr:>5.1f}%')
    print(f'  │  Total PnL    : {t["pnl_pct"].sum():>+6.2f}%')
    print(f'  │  Avg win      : {wins["pnl_pct"].mean():>+6.2f}%' if len(wins) > 0 else '  │  Avg win      :    n/a')
    print(f'  │  Avg loss     : {loss["pnl_pct"].mean():>+6.2f}%' if len(loss) > 0 else '  │  Avg loss     :    n/a')
    print(f'  │  Profit factor: {pf:>5.2f}')
    print(f'  │  Max drawdown : {maxdd:>5.2f}%')
    print(f'  └' + '─'*50)


def plot_results(results_dict):
    if not results_dict:
        return
    colors = ['#2196F3','#4CAF50','#FF9800','#E91E63']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f'Backtest — SMC Scalping  |  Data: {DATA_SOURCE.upper()}  →  Exec: {EXEC_EXCHANGE.upper()}',
        fontsize=13, fontweight='bold')
    ax_price, ax_eq, ax_dd, ax_dist = axes[0,0], axes[0,1], axes[1,0], axes[1,1]
    all_trades = []

    for idx, (symbol, res) in enumerate(results_dict.items()):
        if not res:
            continue
        col = colors[idx % len(colors)]
        df, trades, equity = res['df'], res['trades'], res['equity']
        all_trades.extend(trades)

        if idx == 0:
            ax_price.plot(df.index, df['close'], color=col, lw=0.8, alpha=0.7, label='Close')
            ax_price.plot(df.index, df['ema9'],  color='orange', lw=0.8, ls='--', label='EMA9', alpha=0.7)
            ax_price.plot(df.index, df['ema21'], color='blue',   lw=0.8, ls='--', label='EMA21', alpha=0.7)
            longs  = df[df['long_signal']]
            shorts = df[df['short_signal']]
            ax_price.scatter(longs.index,  longs['close'],  marker='^', color='#4CAF50', s=60, zorder=5, label='Long')
            ax_price.scatter(shorts.index, shorts['close'], marker='v', color='#E91E63', s=60, zorder=5, label='Short')
            ax_price.set_title(f'Price + signals ({symbol})')
            ax_price.legend(fontsize=7)
            ax_price.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
            plt.setp(ax_price.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)
            ax_price.grid(True, alpha=0.2)

        eq_s = pd.Series(equity, index=df.index[:len(equity)])
        ax_eq.plot(eq_s.index, eq_s, color=col, lw=1.2, label=symbol)
        dd = (eq_s - eq_s.cummax()) / eq_s.cummax() * 100
        ax_dd.fill_between(dd.index, dd, 0, alpha=0.35, color=col, label=symbol)
        ax_dd.plot(dd.index, dd, color=col, lw=0.6)

    if all_trades:
        pnls = [t['pnl_pct'] for t in all_trades]
        ax_dist.hist([p for p in pnls if p > 0],  bins=30, color='#4CAF50', alpha=0.7, label=f'Wins')
        ax_dist.hist([p for p in pnls if p <= 0], bins=30, color='#E91E63', alpha=0.7, label=f'Losses')
        ax_dist.axvline(0, color='gray', lw=0.8, ls='--')
        ax_dist.set_title('PnL distribution')
        ax_dist.legend(fontsize=8)
        ax_dist.grid(True, alpha=0.2)

    for ax, title in [(ax_eq, 'Equity curve (base=100)'), (ax_dd, 'Drawdown %')]:
        ax.axhline(0, color='gray', lw=0.6, ls='--')
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)

    plt.tight_layout()
    fname = f'backtest_{DATA_SOURCE}_to_{EXEC_EXCHANGE}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'📊 Chart saved to {fname}')


# ── Run ───────────────────────────────────────────────────────────────────────
print(f'🔬 Running backtest on {DATA_SOURCE.upper()} data...\n')
backtest_results = {}
for symbol, df_hist in historical_data.items():
    result = run_backtest(df_hist, STRATEGY_CONFIG, symbol=symbol)
    backtest_results[symbol] = result
    if result:
        print_report(result['trades'], symbol)

print('\n📈 Generating charts...')
plot_results(backtest_results)

all_trades_flat = [t for r in backtest_results.values() if r for t in r['trades']]
if all_trades_flat:
    total_pnl = sum(t['pnl_pct'] for t in all_trades_flat)
    wr_all    = sum(1 for t in all_trades_flat if t['pnl_pct'] > 0) / len(all_trades_flat) * 100
    print(f'\n─── COMBINED ({DATA_SOURCE.upper()} data) ────────────────────────')
    print(f'  Total trades : {len(all_trades_flat)}')
    print(f'  Win rate     : {wr_all:.1f}%')
    print(f'  Total PnL    : {total_pnl:+.2f}%')
    print(f'────────────────────────────────────────────────────')
print('\n✅ Backtest complete. Review results, then run Phase 5 to start the bot.')

---
## Phase 5 — Live / Paper loop
⚠️ **Only run after reviewing backtest results.**

The live loop uses:
- **`DATA_SOURCE`** to fetch real-time candles each cycle
- **`EXEC_EXCHANGE`** to place orders (or simulate in paper mode)
- **`PAIR_MAP`** to translate data symbols → execution symbols

### To switch modes
```python
# In Cell 2:
TRADING_MODE  = 'paper'    # safe default
TRADING_MODE  = 'live'     # real orders — requires API key

# Example: pull Binance data, execute on Kraken
DATA_SOURCE   = 'binance'
EXEC_EXCHANGE = 'kraken'
PAIR_MAP      = {'BTC/USDT': 'BTC/USD', 'SOL/USDT': 'SOL/USD'}
# Then re-run Cell 2 → Cell 5 → Cell 10 → Cell 11
```

In [ ]:
# ── CELL 10 — Paper ledger + execution helpers ───────────────────────────────

class PaperLedger:
    def __init__(self, equity):
        self.equity    = equity
        self.positions = {}
        self.trade_log = []

    def open_position(self, symbol, direction, price, qty, sl, tp, t_pts, t_off):
        self.equity -= price * qty * (STRATEGY_CONFIG['commission'] / 100)
        self.positions[symbol] = {
            'direction': direction, 'entry': price, 'qty': qty,
            'sl': sl, 'tp': tp, 'trail_points': t_pts, 'trail_offset': t_off}
        print(f'  📥 PAPER {direction.upper():5s} {symbol} @ {price:.4f}'
              f' | SL={sl:.4f}  TP={tp:.4f}  Qty={qty:.6f}')

    def check_exit(self, symbol, high, low):
        if symbol not in self.positions:
            return
        pos = self.positions[symbol]
        sl, tp, d = pos['sl'], pos['tp'], pos['direction']
        if STRATEGY_CONFIG['use_trailing']:
            t, o = pos['trail_points'], pos['trail_offset']
            if d == 'long'  and (high - pos['entry']) >= t: sl = max(sl, high - o)
            elif d == 'short' and (pos['entry'] - low)  >= t: sl = min(sl, low + o)
            pos['sl'] = sl
        tp_hit   = (d == 'long' and high >= tp) or (d == 'short' and low  <= tp)
        stop_hit = (d == 'long' and low  <= sl) or (d == 'short' and high >= sl)
        if not (tp_hit or stop_hit):
            return
        exit_price = tp if tp_hit else sl
        qty  = pos['qty']
        net  = ((exit_price - pos['entry']) if d == 'long' else (pos['entry'] - exit_price)) * qty
        net -= exit_price * qty * (STRATEGY_CONFIG['commission'] / 100)
        self.equity += net
        tag  = 'TP' if tp_hit else 'SL'
        icon = '✅' if tp_hit else '❌'
        print(f'  📤 PAPER {d.upper():5s} {symbol} @ {exit_price:.4f}'
              f' | {icon} {tag}  PnL={net:+.2f} USD  Equity=${self.equity:.2f}')
        self.trade_log.append({
            'symbol': symbol, 'direction': d, 'entry': pos['entry'],
            'exit': exit_price, 'qty': qty, 'result': tag,
            'net_pnl_usd': round(net, 4), 'equity': round(self.equity, 2)})
        del self.positions[symbol]

    def summary(self):
        if not self.trade_log:
            print('No completed trades yet.')
            return None
        df   = pd.DataFrame(self.trade_log)
        wins = df[df['net_pnl_usd'] > 0]
        loss = df[df['net_pnl_usd'] <= 0]
        pf   = abs(wins['net_pnl_usd'].sum() / loss['net_pnl_usd'].sum()) if len(loss) > 0 else float('inf')
        print('\n' + '═'*52)
        print(f'  📊 PAPER SUMMARY  (data:{DATA_SOURCE.upper()} exec:{EXEC_EXCHANGE.upper()})')
        print('═'*52)
        print(f'  Trades       : {len(df)}  (W:{len(wins)} / L:{len(loss)})')
        print(f'  Win rate     : {len(wins)/len(df)*100:.1f}%')
        print(f'  Total PnL    : ${df["net_pnl_usd"].sum():.2f}')
        print(f'  Profit factor: {pf:.2f}')
        print(f'  Equity now   : ${self.equity:.2f}')
        print('═'*52)
        return df


paper_ledger   = PaperLedger(PAPER_EQUITY)
live_positions = {}   # { exec_symbol: { sl_order_id, tp_order_id, ... } }


def compute_qty(exec_symbol, price, sl):
    """Risk-based sizing: qty = (equity × risk%) / stop_distance"""
    equity    = paper_ledger.equity if TRADING_MODE == 'paper' else _live_equity()
    stop_dist = abs(price - sl)
    if stop_dist == 0:
        return 0.0
    qty  = (equity * STRATEGY_CONFIG['risk_percent'] / 100) / stop_dist
    base = exec_symbol.split('/')[0]
    mins = {'BTC': 0.0001, 'XBT': 0.0001, 'ETH': 0.001, 'SOL': 0.5}
    return max(round(qty, 6), mins.get(base, 0.001))

def _live_equity():
    try:
        bal = exec_conn.fetch_balance()
        return sum(float(bal.get(c, {}).get('free', 0) or 0)
                   for c in ['USD', 'USDT'])
    except:
        return 0.0

def safe_cancel(order_id, symbol):
    try:
        exec_conn.cancel_order(order_id, symbol)
    except Exception:
        pass

def check_live_exit(exec_symbol):
    pos = live_positions.get(exec_symbol)
    if not pos:
        return
    try:
        sl_o = exec_conn.fetch_order(pos['sl_order_id'], exec_symbol)
        tp_o = exec_conn.fetch_order(pos['tp_order_id'], exec_symbol)
        if tp_o['status'] == 'closed':
            print(f'  ✅ LIVE TP HIT {exec_symbol}')
            safe_cancel(pos['sl_order_id'], exec_symbol)
            del live_positions[exec_symbol]
        elif sl_o['status'] == 'closed':
            print(f'  ❌ LIVE SL HIT {exec_symbol}')
            safe_cancel(pos['tp_order_id'], exec_symbol)
            del live_positions[exec_symbol]
    except Exception as e:
        print(f'  ⚠️  Exit check error {exec_symbol}: {e}')

def execute_live(exec_symbol, signal):
    if exec_symbol in live_positions:
        check_live_exit(exec_symbol)
        return
    if not (signal['long_signal'] or signal['short_signal']):
        return
    d    = 'long' if signal['long_signal'] else 'short'
    price = signal['close']
    sl   = signal['long_sl']  if d == 'long' else signal['short_sl']
    tp   = signal['long_tp']  if d == 'long' else signal['short_tp']
    qty  = compute_qty(exec_symbol, price, sl)
    side  = 'buy'  if d == 'long' else 'sell'
    xside = 'sell' if d == 'long' else 'buy'
    if qty <= 0:
        return
    print(f'  ⚡ LIVE {d.upper()} {exec_symbol} @ ~{price:.4f}  qty={qty}')
    try:
        entry_o = exec_conn.create_order(exec_symbol, 'market', side, qty)
        sl_o    = exec_conn.create_order(exec_symbol, 'stop_loss', xside, qty, sl,
                                         params={'ordertype': 'stop-loss', 'price': sl})
        tp_o    = exec_conn.create_order(exec_symbol, 'limit', xside, qty, tp)
        live_positions[exec_symbol] = {
            'direction': d, 'qty': qty, 'entry': price, 'sl': sl, 'tp': tp,
            'sl_order_id': sl_o['id'], 'tp_order_id': tp_o['id']}
        print(f'    Entry={entry_o["id"]}  SL={sl_o["id"]}  TP={tp_o["id"]}')
    except ccxt.InsufficientFunds:
        print(f'  ⚠️  Insufficient funds for {exec_symbol}')
    except Exception as e:
        print(f'  ⚠️  Order error {exec_symbol}: {e}')

def execute_paper(exec_symbol, signal, df):
    if exec_symbol in paper_ledger.positions and not df.empty:
        last = df.iloc[-1]
        paper_ledger.check_exit(exec_symbol, float(last['high']), float(last['low']))
        return
    if signal['long_signal'] or signal['short_signal']:
        d  = 'long' if signal['long_signal'] else 'short'
        sl = signal['long_sl']  if d == 'long' else signal['short_sl']
        tp = signal['long_tp']  if d == 'long' else signal['short_tp']
        qty = compute_qty(exec_symbol, signal['close'], sl)
        if qty > 0:
            paper_ledger.open_position(
                exec_symbol, d, signal['close'], qty, sl, tp,
                signal['trail_points'], signal['trail_offset'])

mode_label = '📄 PAPER' if TRADING_MODE == 'paper' else '⚡ LIVE'
print(f'✅ Execution helpers ready')
print(f'   Data: {DATA_SOURCE.upper()}  →  Exec: {EXEC_EXCHANGE.upper()}  |  Mode: {mode_label}')
print(f'   Paper equity reset to ${PAPER_EQUITY:,.2f}')

In [ ]:
# ── CELL 11 — START THE BOT ──────────────────────────────────────────────────
# Each cycle:
#   1. Fetch live candles from DATA_SOURCE  (no auth needed)
#   2. Compute signals
#   3. Map data symbol → exec symbol via PAIR_MAP
#   4. Execute on EXEC_EXCHANGE (paper or live)

if TRADING_MODE == 'live':
    print('⚡ LIVE MODE — real orders on', EXEC_EXCHANGE.upper())
    confirm = input("Type 'YES' to confirm: ")
    if confirm.strip() != 'YES':
        print('Aborted.')
        raise SystemExit
else:
    print('📄 PAPER MODE')

print(f'   Data source : {DATA_SOURCE.upper()}')
print(f'   Exec        : {EXEC_EXCHANGE.upper()}')
print(f'   Data pairs  : {DATA_PAIRS}')
print(f'   Exec pairs  : {EXEC_PAIRS}')
print(f'   Loop every {LOOP_INTERVAL}s — stop with ■ Interrupt Kernel')
print('─'*60)

cycle = 0
try:
    while True:
        cycle += 1
        now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
        print(f'\n{"═"*60}')
        print(f'  CYCLE #{cycle}  —  {now}')
        if TRADING_MODE == 'paper':
            print(f'  💰 Virtual equity: ${paper_ledger.equity:,.2f}')
        print('═'*60)

        for data_symbol in DATA_PAIRS:
            # Map to the corresponding execution symbol
            exec_symbol = PAIR_MAP.get(data_symbol, data_symbol)
            print(f'\n  📊 {data_symbol} (data) → {exec_symbol} (exec)')

            # 1. Fetch live candles from data source
            df = fetch_live_candles(data_symbol)
            if df.empty:
                print('    ⚠️  No data — skipping')
                continue

            # 2. Compute signals on data-source candles
            _, signal = compute_signals(df, STRATEGY_CONFIG)
            print(f"    Close={signal['close']:.4f}  RSI={signal['rsi']:.1f}  "
                  f"ATR={signal['atr']:.6f}  "
                  f"Long={'✅' if signal['long_signal'] else '❌'}  "
                  f"Short={'✅' if signal['short_signal'] else '❌'}")

            # 3. Execute on execution exchange using the mapped symbol
            if TRADING_MODE == 'paper':
                execute_paper(exec_symbol, signal, df)
            else:
                execute_live(exec_symbol, signal)

        if TRADING_MODE == 'paper' and cycle % 10 == 0:
            paper_ledger.summary()

        print(f'\n  ⏱  Sleeping {LOOP_INTERVAL}s...')
        time.sleep(LOOP_INTERVAL)

except KeyboardInterrupt:
    print('\n🛑 Bot stopped.')
    if TRADING_MODE == 'paper':
        paper_ledger.summary()
    elif live_positions:
        print(f'⚠️  Open live positions: {list(live_positions.keys())}')
        print('    Close them manually on your exchange!')

In [ ]:
# ── CELL 12 — View paper trade history ───────────────────────────────────────
from IPython.display import display
trade_df = paper_ledger.summary()
if trade_df is not None:
    def color_pnl(val):
        if isinstance(val, (int, float)):
            return 'color:green;font-weight:bold' if val > 0 else 'color:red;font-weight:bold'
        return ''
    display(trade_df.style.applymap(color_pnl, subset=['net_pnl_usd']))